In [4]:
import warnings; warnings.filterwarnings("ignore")
import torch
import observer

torch.set_grad_enabled(False)
torch.set_default_dtype(torch.double)

ds_train_name = "spiralwaves" # "ks2d_large_data" or "ks2d_small_data"

if ds_train_name == "spiralwaves":
    ds_test_name = "spiralwaves"
    n_pred = 50
    n_obs = 30
    n_eigenvalues = 3000

if ds_train_name in ["ks2d_large_data", "ks2d_small_data"]:
    ds_test_name = "ks2d"
    n_pred = 100
    n_obs = 5
    n_eigenvalues = 1000

C_train = torch.load(f"data/{ds_train_name}_train.pt", weights_only=True)
C_test = torch.load(f"data/{ds_test_name}_test.pt", weights_only=True)

class ConvEDMD(torch.nn.Module):
    def __init__(self, n_obs, X, Y):
        super().__init__()
        self.G = (X.shape[-2], X.shape[-1])
        self.n_obs = n_obs
        self.A_hat = torch.zeros( n_obs, n_obs, *self.G, dtype=torch.complex128)
        self.train(X, Y)

    def train(self, X, Y):
        X_hat = torch.fft.fft2(X).permute(2,3,0,1)
        Y_hat = torch.fft.fft2(Y).permute(2,3,0,1)
        self.A_hat = torch.linalg.lstsq(X_hat, Y_hat, driver="gelsd").solution.transpose(-1,-2)
        self.A_hat = self.A_hat.permute(2,3,0,1)
        self.A = torch.fft.ifft2(self.A_hat)

    def forward(self, x):
        squeezed = len(x.shape) == 3
        if squeezed: x = x.unsqueeze(0)

        x_hat = torch.fft.fft2(x)
        y_hat = torch.einsum("ijkl,bjkl -> bikl", self.A_hat, x_hat)
        y = torch.fft.ifft2(y_hat)

        if squeezed: y = y.squeeze(0)
        return y

    def A_transposed(self):
        # exchange s and s'
        AT = self.A.transpose(0,1)
        # inverts g
        AT = AT.flip(dims=[2,3])
        AT = AT.roll([1,1], dims=[2,3]) 
        return AT
    
    def eigvals_right(self):
        eigvals_A, _ = torch.linalg.eig( self.A_hat.permute(2,3,0,1) )
        eigvals_A = eigvals_A.flatten()
        return eigvals_A
    
    def eigenpairs_left(self, n_eigenvalues):
        AT = self.A_transposed()
        AT_hat = torch.fft.fft2(AT)

        # evals: (*G, n_obs)
        # evecs_hat: (*G, nr_evecs, dim_evecs)
        evals, evecs_hat = torch.linalg.eig( AT_hat.permute(2,3,0,1) )

        # randomly samples eigenvalues and eigenvectors
        I = [ torch.randint(g, size=(n_eigenvalues//self.n_obs,)) for g in self.G ]
        evals = evals[I[0], I[1]].flatten()

        evecs_hat = evecs_hat[I[0], I[1]] # (n_group_samples, n_evecs = n_obs, dim_evecs = n_obs)
        evecs = torch.zeros(n_eigenvalues//self.n_obs, self.n_obs, self.n_obs, *self.G, dtype=torch.complex128)
        for i, (g1,g2) in enumerate(zip(*I)): evecs[i,:,:,g1,g2] = evecs_hat[i,:,:].T
        evecs = torch.fft.ifft2(evecs)
        evecs = evecs.view(-1, self.n_obs, *self.G)
        evecs = evecs / evecs.abs().pow(2).sum(dim=[1,2,3]).pow(0.5).view(-1,1,1,1) # normalize magnitude to 1

        return evals, evecs


torch.manual_seed(0) # <- makes sure that the same random weights are generated for full and convolutional EDMD 
obs = observer.Observer(n_obs=n_obs)
rec = observer.Reconstructor(obs(C_train), C_train)

X, Y = obs(C_train[:-1]), obs(C_train[1:])
K = ConvEDMD(obs.n_obs, X, Y)


loss: 3.975e-03:  48%|████▊     | 48/100 [00:08<00:09,  5.62it/s]


KeyboardInterrupt: 

In [ ]:
eigvals, eigvecs = K.eigenpairs_left(n_eigenvalues)
eigfuncs = torch.einsum( "nijk,mijk -> nm", obs(C_test).to(torch.complex128), eigvecs )
print(f"saving {eigvals.shape[0]} eigenvalues")

torch.save(eigvals, f"generated data/{ds_train_name}_eigvals.pt")
torch.save(eigfuncs, f"generated data/{ds_train_name}_eigfuncs.pt")

saving 990 eigenvalues


In [ ]:
X_test = obs(C_test)

C_pred = torch.zeros(n_pred, *C_test.shape[1:])
X_pred = torch.zeros(n_pred, *X.shape[1:])
X_pred[0] = X_test[0]
C_pred[0] = rec(X_pred[0])
for j in range(n_pred-1):
    X_pred[j+1] = K( X_pred[j] )
    C_pred[j+1] = rec( X_pred[j+1] )

torch.save(C_pred, f"generated data/{ds_train_name}_trajectory_pred.pt")
torch.save(C_test[:n_pred], f"generated data/{ds_train_name}_trajectory_test.pt")